In [ ]:
import matplotlib.pyplot as plt
import os
import scipy.io as scio
import numpy as np
import mne
from antio import read_cnt
from antio.parser import read_triggers
from matplotlib import pyplot as plt
from mne.datasets import testing
from mne.io import read_raw_brainvision
from mne.io import read_raw_ant
from mne.viz import plot_topomap
import glob
from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from mne.stats import f_threshold_mway_rm, f_mway_rm, fdr_correction
import matplotlib.pyplot as plt
import matplotlib
from mne.viz import (plot_ch_adjacency as plot_adj,
                     plot_topomap as plot_topo)

from mne.viz import set_browser_backend
set_browser_backend("qt")

from scipy import stats
from scipy.stats import ttest_rel as ttest

from EEG_MVPA_Package import (
    erp_loader,
    pol_to_car,
    multi_window_topo_plot,
    edit_colomap
)

In [ ]:
def get_epoch_path(root_path,sub_index,blk,type):
    participant_list = os.listdir(root_path)
    data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_'+'-epo.fif')[0]
    return data_path

def get_export_path(export_root_path,root_path,sub_index,blk,type):
    participant_list = os.listdir(root_path)
    data_path = glob.glob(root_path+participant_list[sub_index]+'/'+'*'+blk+'_'+type+'_'+'-epo.fif')[0]
    locker = np.max(np.array([i for i,ch in enumerate(root_path) if ch=='/']))
    locker_end = np.max(np.array([i for i,ch in enumerate(data_path) if ch=='\\']))
    export_path = export_root_path + data_path[(locker+1):locker_end] + '/'
    export_file =  data_path[(locker_end+1):]
    return export_path,export_file

In [ ]:
def set_inf_reref(epoch):
    sphere = mne.make_sphere_model("auto", "auto", epoch.info)
    src = mne.setup_volume_source_space(sphere=sphere, exclude=30.0, pos=5.0)
    forward = mne.make_forward_solution(epoch.info, trans=None, src=src, bem=sphere)
    epoch_reref = epoch.copy().set_eeg_reference("REST", forward=forward)
    return epoch_reref

In [ ]:
raw_eeg_path = 'Data/'
participant_list = os.listdir(raw_eeg_path)
print(participant_list)

blk_list = ['blk-1','blk-2','blk-3']
type_list = ['lowband','highband']

In [ ]:
participant_index = 0

epoch_file_path = get_epoch_path(raw_eeg_path,participant_index,blk_list[1],type_list[0])
print(epoch_file_path)

In [ ]:
epoch = mne.read_epochs(epoch_file_path)
print('\n\n\n',epoch.info,'\n\n\n')
print(len(epoch),epoch.baseline,epoch.tmin)

In [ ]:
participants_index = np.arange(19)
n_participants = len(participants_index)
print(n_participants)

# Infinite Rereferencing

In [ ]:
rereference_export_root_path = 'Data/Inf_Ref/'

i=0
epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
print(epoch_file_path,'\n\n')
print(get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[1]))

In [ ]:
# low band
for i in participants_index:
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[0])
    epoch_all = mne.read_epochs(epoch_file_path)
    epoch_all.del_proj()
    epoch_all = mne.add_reference_channels(epoch_all, ref_channels=["CPz"])
    epoch_all.pick("eeg").set_montage("standard_1020")
    epoch_all = set_inf_reref(epoch_all)

    epoch_export_path,epoch_export_fname = get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[0])
    
    if not os.path.exists(epoch_export_path):
        os.makedirs(epoch_export_path)

    epoch_all.save(epoch_export_path+epoch_export_fname,overwrite=True)
    evoked = epoch_all.average()
    evoked.plot(spatial_colors=True)

#high band
for i in participants_index:
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
    epoch_all = mne.read_epochs(epoch_file_path)
    epoch_all.del_proj()
    epoch_all = mne.add_reference_channels(epoch_all, ref_channels=["CPz"])
    epoch_all.pick("eeg").set_montage("standard_1020")
    epoch_all = set_inf_reref(epoch_all)

    epoch_export_path,epoch_export_fname = get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[1])
    
    if not os.path.exists(epoch_export_path):
        os.makedirs(epoch_export_path)

    epoch_all.save(epoch_export_path+epoch_export_fname,overwrite=True)
    evoked = epoch_all.average()
    evoked.plot(spatial_colors=True)

In [ ]:
i

In [ ]:
#get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[0])
root_path = raw_eeg_path
participant_list = os.listdir(root_path)
print(root_path+participant_list[i]+'/'+'*'+blk_list[1]+'_'+type_list[0]+'_'+'onsetshifting-epo.fif')
glob.glob(root_path+participant_list[i]+'/'+'*'+blk_list[1]+'_'+type_list[0]+'_'+'onsetshifting-epo.fif')

# Mastoid Rereferencing

In [ ]:
rereference_export_root_path = 'C:/Users/yu028288/OneDrive - University of Central Florida/Documents/Python Scripts/SWS_OneShotLearning_2025/Analysis/EEG_SWS/EEG/EEG_SWS_Preprocessed/Mastoid_Reref/'

i=0
epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
print(epoch_file_path,'\n\n')
print(get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[1]))

In [ ]:
# low band
for i in participants_index:
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[0])
    epoch_all = mne.read_epochs(epoch_file_path)
    epoch_all.del_proj()
    epoch_all = mne.add_reference_channels(epoch_all, ref_channels=["CPz"])
    epoch_all.pick("eeg").set_montage("standard_1020")
    epoch_all.set_eeg_reference(ref_channels=['M1', 'M2'])

    epoch_export_path,epoch_export_fname = get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[0])
    
    if not os.path.exists(epoch_export_path):
        os.makedirs(epoch_export_path)

    epoch_all.save(epoch_export_path+epoch_export_fname,overwrite=True)
    evoked = epoch_all.average()
    evoked.plot(spatial_colors=True)

#high band
for i in participants_index:
    epoch_file_path = get_epoch_path(raw_eeg_path,i,blk_list[1],type_list[1])
    epoch_all = mne.read_epochs(epoch_file_path)
    epoch_all.del_proj()
    epoch_all = mne.add_reference_channels(epoch_all, ref_channels=["CPz"])
    epoch_all.pick("eeg").set_montage("standard_1020")
    epoch_all.set_eeg_reference(ref_channels=['M1', 'M2'])

    epoch_export_path,epoch_export_fname = get_export_path(rereference_export_root_path,raw_eeg_path,i,blk_list[1],type_list[1])
    
    if not os.path.exists(epoch_export_path):
        os.makedirs(epoch_export_path)

    epoch_all.save(epoch_export_path+epoch_export_fname,overwrite=True)
    evoked = epoch_all.average()
    evoked.plot(spatial_colors=True)